In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from CBFV import composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import re
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from ax.service.ax_client import AxClient, ObjectiveProperties

c:\Users\Chris\Documents\ML_Project\ML_GFA_env\Lib\site-packages\CBFV\composition.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
class FlexibleNN(nn.Module):
    """
    A flexible neural network with configurable architecture for hyperparameter optimization.
    
    Parameters:
    -----------
    input_dim : int
        Number of input features (CBFV features)
    output_dim : int
        Number of output targets (phase fractions at selected temperatures)
    hidden_layers : list of int
        List of hidden layer sizes, e.g., [256, 128, 64]
    dropout_rate : float
        Dropout probability (0.0 to 1.0)
    dropout_type : str
        Type of dropout: 'standard', 'alpha' (for SELU), or 'none'
    activation : str
        Activation function: 'relu', 'leaky_relu', 'elu', 'selu', 'gelu', 'tanh'
    use_batch_norm : bool
        Whether to use batch normalization
    use_layer_norm : bool
        Whether to use layer normalization (alternative to batch norm)
    weight_decay : float
        L2 regularization strength (applied in optimizer, stored here for reference)
    nf_indices : list of int or None
        Indices of output columns that are phase fractions (NF). 
        These will have softmax applied so they sum to 1.
        If None, no softmax constraint is applied.
    output_columns : list of str or None
        Column names for outputs. If provided, NF columns are auto-detected
        by checking for 'NF_' prefix. Overrides nf_indices if both provided.
    """
    
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_layers=[256, 128, 64],
        dropout_rate=0.2,
        dropout_type='standard',
        activation='relu',
        use_batch_norm=False,
        use_layer_norm=False,
        weight_decay=0.0,
        nf_indices=None,
        output_columns=None
    ):
        super(FlexibleNN, self).__init__()
        
        self.weight_decay = weight_decay  # Store for optimizer configuration
        self.output_dim = output_dim
        
        # Determine NF indices (phase fractions that should sum to 1)
        if output_columns is not None:
            # Auto-detect NF columns from column names
            self.nf_indices = [i for i, col in enumerate(output_columns) if col.startswith('NF_')]
            self.df_indices = [i for i, col in enumerate(output_columns) if col.startswith('DF_')]
        elif nf_indices is not None:
            self.nf_indices = nf_indices
            self.df_indices = [i for i in range(output_dim) if i not in nf_indices]
        else:
            self.nf_indices = []
            self.df_indices = list(range(output_dim))
        
        # Build activation function
        activation_fn = self._get_activation(activation)
        
        # Build dropout layer
        dropout_layer = self._get_dropout(dropout_type, dropout_rate)
        
        # Build the network layers
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_layers:
            # Linear layer
            layers.append(nn.Linear(prev_dim, hidden_dim))
            
            # Normalization (batch norm or layer norm, not both)
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(hidden_dim))
            elif use_layer_norm:
                layers.append(nn.LayerNorm(hidden_dim))
            
            # Activation
            layers.append(activation_fn())
            
            # Dropout
            if dropout_layer is not None:
                layers.append(dropout_layer(dropout_rate))
            
            prev_dim = hidden_dim
        
        # Output layer (no activation, dropout, or normalization)
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def _get_activation(self, activation):
        activations = {
            'relu': nn.ReLU,
            'leaky_relu': lambda: nn.LeakyReLU(0.1),
            'elu': nn.ELU,
            'selu': nn.SELU,
            'gelu': nn.GELU,
            'tanh': nn.Tanh,
            'sigmoid': nn.Sigmoid
        }
        if activation not in activations:
            raise ValueError(f"Unknown activation: {activation}. Choose from {list(activations.keys())}")
        return activations[activation]
    
    def _get_dropout(self, dropout_type, dropout_rate):
        if dropout_type == 'none' or dropout_rate == 0:
            return None
        elif dropout_type == 'standard':
            return nn.Dropout
        elif dropout_type == 'alpha':
            return nn.AlphaDropout  # For use with SELU activation
        else:
            raise ValueError(f"Unknown dropout type: {dropout_type}. Choose from ['standard', 'alpha', 'none']")
    
    def forward(self, x):
        raw_output = self.network(x)
        
        # If no NF indices, return raw output
        if len(self.nf_indices) == 0:
            return raw_output
        
        # Apply softmax to NF (phase fraction) outputs so they sum to 1
        output = raw_output.clone()
        
        # Extract NF outputs and apply softmax
        nf_outputs = raw_output[:, self.nf_indices]
        nf_normalized = torch.softmax(nf_outputs, dim=1)
        
        # Put normalized NF values back
        output[:, self.nf_indices] = nf_normalized
        
        return output
    
    def get_optimizer(self, optimizer_type='adam', lr=1e-3):
        """
        Get an optimizer with the configured weight decay (L2 regularization).
        """
        optimizers = {
            'adam': lambda: torch.optim.Adam(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'adamw': lambda: torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=self.weight_decay),
            'sgd': lambda: torch.optim.SGD(self.parameters(), lr=lr, weight_decay=self.weight_decay, momentum=0.9),
            'rmsprop': lambda: torch.optim.RMSprop(self.parameters(), lr=lr, weight_decay=self.weight_decay)
        }
        if optimizer_type not in optimizers:
            raise ValueError(f"Unknown optimizer: {optimizer_type}. Choose from {list(optimizers.keys())}")
        return optimizers[optimizer_type]()


def train_epoch(model, train_loader, optimizer, criterion, device):
    """
    Train the model for one epoch.
    
    Returns:
    --------
    float : Average training loss for the epoch
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def evaluate_epoch(model, val_loader, criterion, device):
    """
    Evaluate the model on validation data.
    
    Returns:
    --------
    tuple : (average loss, predictions, targets)
    """
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            batch_size = X_batch.size(0)
            
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            
            # Weight loss by batch size for correct averaging
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            
            all_predictions.append(predictions.cpu())
            all_targets.append(y_batch.cpu())
    
    # Weighted average loss (accounts for different batch sizes)
    avg_loss = total_loss / total_samples
    all_predictions = torch.cat(all_predictions, dim=0)
    all_targets = torch.cat(all_targets, dim=0)
    
    return avg_loss, all_predictions, all_targets


def train_model(
    model,
    X_train, y_train,
    X_val, y_val,
    epochs=100,
    batch_size=32,
    optimizer_type='adam',
    lr=1e-3,
    criterion=None,
    early_stopping_patience=None,
    verbose=True,
    device=None
):
    """
    Train the model for a specified number of epochs.
    
    Parameters:
    -----------
    model : FlexibleNN
        The neural network model
    X_train, y_train : array-like
        Training data
    X_val, y_val : array-like
        Validation data
    epochs : int
        Number of training epochs
    batch_size : int
        Batch size for training
    optimizer_type : str
        Type of optimizer ('adam', 'adamw', 'sgd', 'rmsprop')
    lr : float
        Learning rate
    criterion : nn.Module
        Loss function (default: MSELoss)
    early_stopping_patience : int or None
        Stop training if val loss doesn't improve for this many epochs
    verbose : bool
        Whether to print progress
    device : str or None
        Device to train on ('cuda', 'mps', 'cpu', or None for auto-detect)
    
    Returns:
    --------
    dict : Training history with train_losses, val_losses, best_epoch
    """
    # Auto-detect device
    if device is None:
        if torch.cuda.is_available():
            device = 'cuda'
        elif torch.backends.mps.is_available():
            device = 'mps'
        else:
            device = 'cpu'
    
    device = torch.device(device)
    model = model.to(device)
    
    # Default criterion
    if criterion is None:
        criterion = nn.MSELoss()
    
    # Convert data to tensors
    X_train_t = torch.FloatTensor(X_train.values if hasattr(X_train, 'values') else X_train)
    y_train_t = torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train)
    X_val_t = torch.FloatTensor(X_val.values if hasattr(X_val, 'values') else X_val)
    y_val_t = torch.FloatTensor(y_val.values if hasattr(y_val, 'values') else y_val)
    
    # Create data loaders
    train_dataset = TensorDataset(X_train_t, y_train_t)
    val_dataset = TensorDataset(X_val_t, y_val_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Get optimizer
    optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
    
    # Training history
    history = {
        'train_losses': [],
        'val_losses': [],
        'best_epoch': 0,
        'best_val_loss': float('inf')
    }
    
    # Early stopping
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(epochs):
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        
        # Evaluate
        val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
        
        history['train_losses'].append(train_loss)
        history['val_losses'].append(val_loss)
        
        # Track best model
        if val_loss < history['best_val_loss']:
            history['best_val_loss'] = val_loss
            history['best_epoch'] = epoch
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Verbose output
        if verbose and (epoch % 10 == 0 or epoch == epochs - 1):
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
        
        # Early stopping
        if early_stopping_patience and patience_counter >= early_stopping_patience:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}. Best epoch: {history['best_epoch']+1}")
            break
    
    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return history

In [ ]:
def fourier_features(T, n_freqs=6):
    """
    Generate Fourier features for temperature values.
    
    Args:
        T: 1D tensor of temperature values (shape: [N])
        n_freqs: number of frequency components (produces 2*n_freqs features)
    
    Returns:
        2D tensor of shape [N, 2*n_freqs] where each row is the Fourier encoding
    """
    # Ensure T is 2D with shape [N, 1] for broadcasting
    if T.dim() == 1:
        T = T.unsqueeze(1)
    
    feats = []
    for k in range(n_freqs):
        freq = 2**k
        feats.append(torch.sin(2 * torch.pi * freq * T))
        feats.append(torch.cos(2 * torch.pi * freq * T))
    
    # Stack along dim=1 to get shape [N, 2*n_freqs]
    return torch.cat(feats, dim=1)

In [ ]:
#pick which flattened dataset to reshape for F(T) surrogate modeling
calphed_data = pd.read_csv(r'Data/F(Composition)_Data/calphad_alloys_train_opt.csv')

KeyboardInterrupt: 

In [4]:

# Extract unique temperatures from column names
temp_pattern = re.compile(r'_T(\d+)C$')
temperatures = set()
for col in calphed_data.columns:
    match = temp_pattern.search(col)
    if match:
        temperatures.add(int(match.group(1)))

temperatures = sorted(temperatures)

# Extract unique phases (DF and NF prefixes)
phase_pattern = re.compile(r'^(DF|NF)_(.+?)_T\d+C$')
phases = set()
for col in calphed_data.columns:
    match = phase_pattern.match(col)
    if match:
        phases.add((match.group(1), match.group(2)))  # (prefix, phase_name)

phases = sorted(phases)

# Create new dataframe with alloy_string and temperature as rows
rows = []
for idx, row in calphed_data.iterrows():
    alloy = row['alloy_string']
    for temp in temperatures:
        new_row = {'alloy_string': alloy, 'temperature': temp}
        for prefix, phase in phases:
            col_name = f'{prefix}_{phase}_T{temp}C'
            if col_name in calphed_data.columns:
                new_col_name = f'{prefix}_{phase}'
                new_row[new_col_name] = row[col_name]
        rows.append(new_row)

# Create the reshaped dataframe
df_reshaped = pd.DataFrame(rows)

# Reorder columns to have alloy_string, temperature first, then sorted DF and NF columns
cols = ['alloy_string', 'temperature']
other_cols = [c for c in df_reshaped.columns if c not in cols]
other_cols.sort()
df_reshaped = df_reshaped[cols + other_cols]

print(f"Original shape: {calphed_data.shape}")
print(f"Reshaped shape: {df_reshaped.shape}")
print(f"Number of temperatures: {len(temperatures)}")
print(f"Temperatures: {temperatures[:5]}... to ...{temperatures[-5:]}")
df_reshaped.head(10)

NameError: name 'calphed_data' is not defined

In [ ]:
#choose where to save the reshaped dataset for F(T) surrogate modeling
df_reshaped.to_csv(r'Data/F(T)_Data/calphad_alloys_train_opt_reshaped.csv', index=False)

In [ ]:
#load the reshaped dataset to confirm it was saved correctly
df_reshaped = pd.read_csv(r'Data/F(T)_Data/calphad_alloys_train_opt_reshaped.csv')
df_reshaped.head()

,alloy_string,temperature,DF_AG2CA,DF_AG3BE8,DF_AG3CA5,DF_AG3MG,DF_AG7CA2,DF_AG9CA2,DF_AGCA,DF_AGCA3,...,NF_YSI2_H,NF_YSI2_R,NF_ZINCBLENDE_B3,NF_ZR2SI,NF_ZR3SI,NF_ZR3SI2,NF_ZR5SI3,NF_ZR5SI4,NF_ZRSI,NF_ZRSI2
0,B22.00Co4.00Fe68.00Y6.00,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,B22.00Co4.00Fe68.00Y6.00,50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B22.00Co4.00Fe68.00Y6.00,100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,B22.00Co4.00Fe68.00Y6.00,150,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,B22.00Co4.00Fe68.00Y6.00,200,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Extract alloy strings and create formula dataframe
formula_df = pd.DataFrame({'formula': df_reshaped['alloy_string']})
print(f"Formula dataframe shape: {formula_df.shape}")
formula_df.head()

Formula dataframe shape: (44982, 1)


,formula
0,B22.00Co4.00Fe68.00Y6.00
1,B22.00Co4.00Fe68.00Y6.00
2,B22.00Co4.00Fe68.00Y6.00
3,B22.00Co4.00Fe68.00Y6.00
4,B22.00Co4.00Fe68.00Y6.00


In [ ]:
# Generate Composition-Based Feature Vectors using CBFV
# CBFV expects 'formula' column and optionally a 'target' column
# Add a dummy target for featurization (we'll drop it after)
formula_df['target'] = 0

# Generate CBFVs using the magpie element property database
X, y, formulae, skipped = composition.generate_features(formula_df, elem_prop='magpie')
print(f"CBFV features shape: {X.shape}")
print(f"Number of skipped formulas: {len(skipped)}")
X.head()

Processing Input Data:   0%|          | 0/44982 [00:00<?, ?it/s]

Processing Input Data: 100%|██████████| 44982/44982 [00:01<00:00, 42209.47it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 44982/44982 [00:01<00:00, 31418.46it/s]


	Creating Pandas Objects...
CBFV features shape: (44982, 132)
Number of skipped formulas: 0


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,mode_NValence,mode_NsUnfilled,mode_NpUnfilled,mode_NdUnfilled,mode_NfUnfilled,mode_NUnfilled,mode_GSvolume_pa,mode_GSbandgap,mode_GSmagmom,mode_SpaceGroupNumber
0,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
1,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
2,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
3,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0
4,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,8.0,0.0,0.0,4.0,0.0,4.0,10.73,0.0,2.110663,229.0


In [ ]:
fourier_Temp_features = fourier_features(torch.tensor(df_reshaped['temperature'].values, dtype=torch.float32))
fourier_Temp_features_df = pd.DataFrame(
    fourier_Temp_features.numpy(),
    columns=[f'fourier_{i}' for i in range(fourier_Temp_features.shape[1])],
    index=df_reshaped.index
)

In [ ]:
#change depending on which temp features to use

X_combined = pd.concat([X, fourier_Temp_features_df], axis=1)
print(f"X shape: {X_combined.shape}")
X_combined.head()

X shape: (44982, 144)


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,fourier_2,fourier_3,fourier_4,fourier_5,fourier_6,fourier_7,fourier_8,fourier_9,fourier_10,fourier_11
0,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.0,0.000000,1.000000
1,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000012,1.0,0.000024,1.0,0.000047,1.0,0.000094,1.0,0.000188,1.000000
2,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000024,1.0,0.000047,1.0,0.000094,1.0,0.000188,1.0,0.000376,1.000000
3,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000096,1.0,0.000193,1.0,0.000385,1.0,0.000771,1.0,0.001541,0.999999
4,22.2,56.28,48.044699,1926.7,8.84,3.62,124.68,1.8416,2.0,0.22,...,0.000047,1.0,0.000094,1.0,0.000188,1.0,0.000376,1.0,0.000753,1.000000


In [ ]:
# Create 5 CV groups by clustering similar alloys together
# Standardize features for clustering
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

# Use KMeans to group similar alloys into 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cv_groups = kmeans.fit_predict(X_scaled)

# Add group assignments to a dataframe for reference
cv_group_df = pd.DataFrame({
    'formula': formulae,
    'cv_group': cv_groups
})

print("CV Group distribution:")
print(cv_group_df['cv_group'].value_counts().sort_index())
print(f"\nTotal samples: {len(cv_groups)}")
cv_group_df.head(10)

CV Group distribution:
cv_group
0     7599
1    16014
2     6375
3    10200
4     4794
Name: count, dtype: int64

Total samples: 44982


,formula,cv_group
0,B22.00Co4.00Fe68.00Y6.00,3
1,B22.00Co4.00Fe68.00Y6.00,3
2,B22.00Co4.00Fe68.00Y6.00,3
3,B22.00Co4.00Fe68.00Y6.00,3
4,B22.00Co4.00Fe68.00Y6.00,3
5,B22.00Co4.00Fe68.00Y6.00,3
6,B22.00Co4.00Fe68.00Y6.00,3
7,B22.00Co4.00Fe68.00Y6.00,3
8,B22.00Co4.00Fe68.00Y6.00,3
9,B22.00Co4.00Fe68.00Y6.00,3


In [ ]:
#Create the y which is the phase information
y = df_reshaped.drop(columns=['alloy_string', 'temperature'])
print(f"Phase information shape: {y.shape}")
y.head()

Phase information shape: (44982, 1608)


,DF_AG2CA,DF_AG3BE8,DF_AG3CA5,DF_AG3MG,DF_AG7CA2,DF_AG9CA2,DF_AGCA,DF_AGCA3,DF_AGCD_ETA,DF_AGIN2,...,NF_YSI2_H,NF_YSI2_R,NF_ZINCBLENDE_B3,NF_ZR2SI,NF_ZR3SI,NF_ZR3SI2,NF_ZR5SI3,NF_ZR5SI4,NF_ZRSI,NF_ZRSI2
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
def evaluate_parameters_nn(parameters, batch_size=32, epochs=1000, verbose=True):
    
    # Extract hyperparameters from parameters dict with defaults
    batch_size = parameters.get('batch_size', batch_size)
    hidden_layers = parameters.get('hidden_layers', [256, 128, 64])
    dropout_rate = parameters.get('dropout_rate', 0.2)
    dropout_type = parameters.get('dropout_type', 'standard')
    activation = parameters.get('activation', 'relu')
    use_batch_norm = parameters.get('use_batch_norm', False)
    use_layer_norm = parameters.get('use_layer_norm', False)
    weight_decay = parameters.get('weight_decay', 0.0)
    optimizer_type = parameters.get('optimizer_type', 'adam')
    lr = parameters.get('lr', 1e-3)
    early_stopping_patience = parameters.get('early_stopping_patience', 10)

    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")

    #copy the x and y data
    y_data = y.copy()
    x_data = X_combined.copy()

    # Get input and output dimensions
    input_dim = x_data.shape[1]
    output_dim = y_data.shape[1]
    print(f"Input dim: {input_dim}, Output dim: {output_dim}")

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):  # 0-4 for 5 groups from KMeans
        print(f"\n{'='*50}")
        print(f"Fold {test_group}")
        print('='*50)
        
        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data in train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]

        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]
        
        # Create train and validation data
        X_train, X_val, y_train, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Scale the x data
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

        #scale the y data
        y_scaler = StandardScaler()
        y_train_scaled = pd.DataFrame(y_scaler.fit_transform(y_train), columns=y_train.columns, index=y_train.index)
        y_val_scaled = pd.DataFrame(y_scaler.transform(y_val), columns=y_val.columns, index=y_val.index)
        y_test_scaled = pd.DataFrame(y_scaler.transform(y_test), columns=y_test.columns, index=y_test.index)

        # Convert DataFrames to PyTorch tensors
        X_train_t = torch.FloatTensor(X_train_scaled.values)
        y_train_t = torch.FloatTensor(y_train_scaled.values)
        X_val_t = torch.FloatTensor(X_val_scaled.values)
        y_val_t = torch.FloatTensor(y_val_scaled.values)
        X_test_t = torch.FloatTensor(x_test_scaled.values)
        y_test_t = torch.FloatTensor(y_test_scaled.values)

        # Create TensorDatasets
        train_dataset = TensorDataset(X_train_t, y_train_t)
        val_dataset = TensorDataset(X_val_t, y_val_t)
        test_dataset = TensorDataset(X_test_t, y_test_t)

        # Create DataLoaders
        # drop_last=True for train_loader to avoid batch size of 1 (causes BatchNorm to fail)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        # Create the model with output_columns to auto-detect NF columns for softmax
        model = FlexibleNN(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            dropout_type=dropout_type,
            activation=activation,
            use_batch_norm=use_batch_norm,
            use_layer_norm=use_layer_norm,
            weight_decay=weight_decay,
            output_columns=y_data.columns.tolist()  # Auto-detect NF columns for softmax constraint
        ).to(device)

        # Get optimizer and criterion
        optimizer = model.get_optimizer(optimizer_type=optimizer_type, lr=lr)
        criterion = nn.MSELoss()

        # Training loop
        best_val_loss = float('inf')
        best_model_state = None
        patience_counter = 0
        train_losses = []
        val_losses = []

        for epoch in range(epochs):
            # Train
            train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
            
            # Evaluate on validation
            val_loss, _, _ = evaluate_epoch(model, val_loader, criterion, device)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            # Track best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Verbose output
            if verbose and (epoch % 20 == 0 or epoch == epochs - 1):
                print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
            
            # Early stopping
            if early_stopping_patience and patience_counter >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Restore best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            model.to(device)

        # Evaluate on test set (scaled)
        test_loss_scaled, test_preds_scaled, test_targets_scaled = evaluate_epoch(model, test_loader, criterion, device)
        
        # Inverse transform predictions and targets to original scale
        test_preds_original = y_scaler.inverse_transform(test_preds_scaled.numpy())
        test_targets_original = y_scaler.inverse_transform(test_targets_scaled.numpy())
        
        # Get column names for separating DF (Driving Force) vs NF (Phase Fraction)
        col_names = y_data.columns.tolist()
        df_cols = [i for i, col in enumerate(col_names) if col.startswith('DF_')]
        nf_cols = [i for i, col in enumerate(col_names) if col.startswith('NF_')]
        
        # Calculate overall RMSE and MAE on original scale (all values)
        mse_original = np.mean((test_preds_original - test_targets_original) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds_original - test_targets_original))
        
        # Calculate metrics for non-zero targets only
        # Use a small threshold to handle floating point precision
        nonzero_threshold = 1e-6
        
        # Overall non-zero metrics
        nonzero_mask = np.abs(test_targets_original) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds_original[nonzero_mask] - test_targets_original[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Driving Force columns (DF_) - non-zero only
        if df_cols:
            df_preds = test_preds_original[:, df_cols]
            df_targets = test_targets_original[:, df_cols]
            df_nonzero_mask = np.abs(df_targets) > nonzero_threshold
            if np.any(df_nonzero_mask):
                df_errors = df_preds[df_nonzero_mask] - df_targets[df_nonzero_mask]
                df_rmse = np.sqrt(np.mean(df_errors ** 2))
                df_mae = np.mean(np.abs(df_errors))
                n_df_nonzero = np.sum(df_nonzero_mask)
            else:
                df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        else:
            df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Phase Fraction columns (NF_) - non-zero only
        if nf_cols:
            nf_preds = test_preds_original[:, nf_cols]
            nf_targets = test_targets_original[:, nf_cols]
            nf_nonzero_mask = np.abs(nf_targets) > nonzero_threshold
            if np.any(nf_nonzero_mask):
                nf_errors = nf_preds[nf_nonzero_mask] - nf_targets[nf_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(nf_errors ** 2))
                nf_mae = np.mean(np.abs(nf_errors))
                n_nf_nonzero = np.sum(nf_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        
        print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
        print(f"Driving Force (DF) - RMSE: {df_rmse:.4f}, MAE: {df_mae:.4f}  [{n_df_nonzero:,} non-zero values]")
        print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'best_val_loss': best_val_loss,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'df_rmse': df_rmse,
            'df_mae': df_mae,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae,
            'train_losses': train_losses,
            'val_losses': val_losses
        })
        
        all_test_predictions.append(torch.FloatTensor(test_preds_original))
        all_test_targets.append(torch.FloatTensor(test_targets_original))

    # Summary across all folds
    print(f"\n{'='*50}")
    print("Cross-Validation Summary (non-zero targets only)")
    print('='*50)
    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])
    
    # Separate metrics for DF and NF (already non-zero)
    avg_df_rmse = np.mean([r['df_rmse'] for r in fold_results])
    avg_df_mae = np.mean([r['df_mae'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    
    print(f"Overall (all values):")
    print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
    print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
    print(f"\nOverall (non-zero only):")
    print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
    print(f"\nDriving Force (DF, non-zero only):")
    print(f"  Avg RMSE: {avg_df_rmse:.4f}, MAE: {avg_df_mae:.4f}")
    print(f"\nPhase Fraction (NF, non-zero only):")
    print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_df_rmse': avg_df_rmse,
        'avg_df_mae': avg_df_mae,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets
    }
    
    
    
    

In [ ]:
def evaluate_parameters_xgb(parameters, verbose=True):
    
    # Extract XGBoost hyperparameters from parameters dict with defaults
    n_estimators = parameters.get('n_estimators', 500)
    max_depth = parameters.get('max_depth', 6)
    learning_rate = parameters.get('learning_rate', 0.1)
    subsample = parameters.get('subsample', 0.8)
    colsample_bytree = parameters.get('colsample_bytree', 0.8)
    min_child_weight = parameters.get('min_child_weight', 1)
    reg_alpha = parameters.get('reg_alpha', 0.0)
    reg_lambda = parameters.get('reg_lambda', 1.0)
    gamma = parameters.get('gamma', 0.0)
    early_stopping_rounds = parameters.get('early_stopping_rounds', 20)

    # Copy the x and y data
    y_data = y.copy()
    x_data = X_combined.copy()

    # Get column names for separating DF (Driving Force) vs NF (Phase Fraction)
    col_names = y_data.columns.tolist()
    df_cols = [i for i, col in enumerate(col_names) if col.startswith('DF_')]
    nf_cols = [i for i, col in enumerate(col_names) if col.startswith('NF_')]

    # Store results for each fold
    fold_results = []
    all_test_predictions = []
    all_test_targets = []

    for test_group in range(5):  # 0-4 for 5 groups from KMeans
        print(f"\n{'='*50}")
        print(f"Fold {test_group}")
        print('='*50)
        
        # Get the split indexes for training and testing
        train_idx = cv_group_df['cv_group'] != test_group
        test_idx = cv_group_df['cv_group'] == test_group

        # Split the x and y data in train and test
        x_train = x_data.loc[train_idx]
        x_test = x_data.loc[test_idx]

        y_train = y_data.loc[train_idx]
        y_test = y_data.loc[test_idx]
        
        # Create train and validation data (for early stopping)
        X_train, X_val, y_train_split, y_val = train_test_split(
            x_train, y_train, test_size=0.2, random_state=42
        )

        print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(x_test)}")

        # Scale the x data
        scaler = StandardScaler()
        X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
        X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)
        x_test_scaled = pd.DataFrame(scaler.transform(x_test), columns=x_test.columns, index=x_test.index)

        # Train one XGBRegressor per output column (manual multi-output)
        # This avoids the MultiOutputRegressor eval_set bug where full multi-column
        # y_val is passed to each single-output estimator
        estimators = []
        best_iters = []
        output_cols = y_train_split.columns.tolist()
        
        for col_idx, col_name in enumerate(output_cols):
            xgb_model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                min_child_weight=min_child_weight,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                gamma=gamma,
                random_state=42,
                n_jobs=-1,
                verbosity=0,
                early_stopping_rounds=early_stopping_rounds,
                eval_metric='rmse',
                device='cuda',
                tree_method='hist',
            )
            
            xgb_model.fit(
                X_train_scaled, y_train_split[col_name],
                eval_set=[(X_val_scaled, y_val[col_name])],
                verbose=False,
            )
            
            estimators.append(xgb_model)
            best_iters.append(xgb_model.best_iteration)

        if verbose:
            print(f"Best iterations (min/mean/max): {min(best_iters)}/{np.mean(best_iters):.0f}/{max(best_iters)}")

        # Predict on test set (stack individual predictions)
        test_preds_original = np.column_stack([
            est.predict(x_test_scaled) for est in estimators
        ])
        test_targets_original = y_test.values

        # Calculate overall RMSE and MAE on original scale (all values)
        mse_original = np.mean((test_preds_original - test_targets_original) ** 2)
        rmse = np.sqrt(mse_original)
        mae = np.mean(np.abs(test_preds_original - test_targets_original))
        
        # Calculate metrics for non-zero targets only
        nonzero_threshold = 1e-6
        
        # Overall non-zero metrics
        nonzero_mask = np.abs(test_targets_original) > nonzero_threshold
        if np.any(nonzero_mask):
            nonzero_errors = test_preds_original[nonzero_mask] - test_targets_original[nonzero_mask]
            rmse_nonzero = np.sqrt(np.mean(nonzero_errors ** 2))
            mae_nonzero = np.mean(np.abs(nonzero_errors))
            n_nonzero = np.sum(nonzero_mask)
        else:
            rmse_nonzero, mae_nonzero, n_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Driving Force columns (DF_) - non-zero only
        if df_cols:
            df_preds = test_preds_original[:, df_cols]
            df_targets = test_targets_original[:, df_cols]
            df_nonzero_mask = np.abs(df_targets) > nonzero_threshold
            if np.any(df_nonzero_mask):
                df_errors = df_preds[df_nonzero_mask] - df_targets[df_nonzero_mask]
                df_rmse = np.sqrt(np.mean(df_errors ** 2))
                df_mae = np.mean(np.abs(df_errors))
                n_df_nonzero = np.sum(df_nonzero_mask)
            else:
                df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        else:
            df_rmse, df_mae, n_df_nonzero = 0.0, 0.0, 0
        
        # Calculate metrics for Phase Fraction columns (NF_) - non-zero only
        if nf_cols:
            nf_preds = test_preds_original[:, nf_cols]
            nf_targets = test_targets_original[:, nf_cols]
            nf_nonzero_mask = np.abs(nf_targets) > nonzero_threshold
            if np.any(nf_nonzero_mask):
                nf_errors = nf_preds[nf_nonzero_mask] - nf_targets[nf_nonzero_mask]
                nf_rmse = np.sqrt(np.mean(nf_errors ** 2))
                nf_mae = np.mean(np.abs(nf_errors))
                n_nf_nonzero = np.sum(nf_nonzero_mask)
            else:
                nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        else:
            nf_rmse, nf_mae, n_nf_nonzero = 0.0, 0.0, 0
        
        print(f"Overall (all)      - RMSE: {rmse:.4f}, MAE: {mae:.4f}")
        print(f"Overall (non-zero) - RMSE: {rmse_nonzero:.4f}, MAE: {mae_nonzero:.4f}  [{n_nonzero:,} values]")
        print(f"Driving Force (DF) - RMSE: {df_rmse:.4f}, MAE: {df_mae:.4f}  [{n_df_nonzero:,} non-zero values]")
        print(f"Phase Fraction (NF) - RMSE: {nf_rmse:.4f}, MAE: {nf_mae:.4f}  [{n_nf_nonzero:,} non-zero values]")

        # Store fold results
        fold_results.append({
            'fold': test_group,
            'test_loss': mse_original,
            'test_rmse': rmse,
            'test_mae': mae,
            'rmse_nonzero': rmse_nonzero,
            'mae_nonzero': mae_nonzero,
            'df_rmse': df_rmse,
            'df_mae': df_mae,
            'nf_rmse': nf_rmse,
            'nf_mae': nf_mae,
        })
        
        all_test_predictions.append(test_preds_original)
        all_test_targets.append(test_targets_original)

    # Summary across all folds
    print(f"\n{'='*50}")
    print("Cross-Validation Summary (non-zero targets only)")
    print('='*50)
    avg_test_loss = np.mean([r['test_loss'] for r in fold_results])
    avg_test_rmse = np.mean([r['test_rmse'] for r in fold_results])
    avg_test_mae = np.mean([r['test_mae'] for r in fold_results])
    avg_rmse_nonzero = np.mean([r['rmse_nonzero'] for r in fold_results])
    avg_mae_nonzero = np.mean([r['mae_nonzero'] for r in fold_results])
    std_test_loss = np.std([r['test_loss'] for r in fold_results])
    
    # Separate metrics for DF and NF (already non-zero)
    avg_df_rmse = np.mean([r['df_rmse'] for r in fold_results])
    avg_df_mae = np.mean([r['df_mae'] for r in fold_results])
    avg_nf_rmse = np.mean([r['nf_rmse'] for r in fold_results])
    avg_nf_mae = np.mean([r['nf_mae'] for r in fold_results])
    
    print(f"Overall (all values):")
    print(f"  Avg Test MSE: {avg_test_loss:.4f} (+/- {std_test_loss:.4f})")
    print(f"  Avg RMSE: {avg_test_rmse:.4f}, MAE: {avg_test_mae:.4f}")
    print(f"\nOverall (non-zero only):")
    print(f"  Avg RMSE: {avg_rmse_nonzero:.4f}, MAE: {avg_mae_nonzero:.4f}")
    print(f"\nDriving Force (DF, non-zero only):")
    print(f"  Avg RMSE: {avg_df_rmse:.4f}, MAE: {avg_df_mae:.4f}")
    print(f"\nPhase Fraction (NF, non-zero only):")
    print(f"  Avg RMSE: {avg_nf_rmse:.4f}, MAE: {avg_nf_mae:.4f}")

    return {
        'fold_results': fold_results,
        'avg_test_loss': avg_test_loss,
        'avg_test_rmse': avg_test_rmse,
        'avg_test_mae': avg_test_mae,
        'avg_rmse_nonzero': avg_rmse_nonzero,
        'avg_mae_nonzero': avg_mae_nonzero,
        'avg_df_rmse': avg_df_rmse,
        'avg_df_mae': avg_df_mae,
        'avg_nf_rmse': avg_nf_rmse,
        'avg_nf_mae': avg_nf_mae,
        'all_predictions': all_test_predictions,
        'all_targets': all_test_targets
    }

In [36]:
parameters = {
    'hidden_layers': [512, 1024],
    'use_batch_norm': True,
    'weight_decay': 1e-4,
    'lr': 1e-3,
    'early_stopping_patience': 15
}
results = evaluate_parameters_nn(parameters)

Using device: mps
Input dim: 133, Output dim: 1608

Fold 0
Train: 27825, Val: 6957, Test: 10200
Epoch 1/1000 - Train Loss: 0.370060 - Val Loss: 0.298181
Epoch 21/1000 - Train Loss: 0.275926 - Val Loss: 0.270464
Early stopping at epoch 35
Overall (all)      - RMSE: 2.8000, MAE: 0.6882
Overall (non-zero) - RMSE: 6.9376, MAE: 3.8382  [695,272 values]
Driving Force (DF) - RMSE: 7.0511, MAE: 3.9563  [673,018 non-zero values]
Phase Fraction (NF) - RMSE: 0.3351, MAE: 0.2660  [22,254 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Epoch 1/1000 - Train Loss: 0.380447 - Val Loss: 0.333415
Epoch 21/1000 - Train Loss: 0.279063 - Val Loss: 0.270137
Early stopping at epoch 31
Overall (all)      - RMSE: 1.4986, MAE: 0.3088
Overall (non-zero) - RMSE: 5.8948, MAE: 2.9751  [1,235,656 values]
Driving Force (DF) - RMSE: 5.9624, MAE: 3.0364  [1,207,679 non-zero values]
Phase Fraction (NF) - RMSE: 0.4088, MAE: 0.3308  [27,977 non-zero values]

Fold 2
Train: 29906, Val: 7477, Test: 7599
Epoch 1

In [16]:
# Define architecture presets mapping (all expanding for large output dim)
ARCHITECTURE_PRESETS = {
    # Single layer - direct expansion
    "single_256": [256],
    "single_512": [512],
    "single_1024": [1024],
    "single_2048": [2048],
    # Two layers - expanding
    "expand_2L_small": [256, 512],
    "expand_2L_medium": [512, 1024],
    "expand_2L_large": [1024, 2048],
    "expand_2L_xlarge": [512, 2048],
    # Three layers - expanding
    "expand_3L_small": [256, 512, 1024],
    "expand_3L_medium": [512, 1024, 2048],
    "expand_3L_large": [256, 1024, 2048],
    "expand_3L_gradual": [384, 768, 1536],
    # Four layers - expanding
    "expand_4L_small": [256, 512, 1024, 2048],
    "expand_4L_medium": [512, 768, 1024, 2048],
    "expand_4L_large": [256, 512, 1024, 4096],
    # Constant width (also good for large outputs)
    "constant_512": [512, 512],
    "constant_1024": [1024, 1024],
    "constant_2048": [2048, 2048],
    "constant_1024_3L": [1024, 1024, 1024],
}

ax_client = AxClient()
ax_client.create_experiment(
    name="NN opt CBFV f(T) Calphed",
    parameters=[
        {
            "name": "architecture",
            "type": "choice",
            "values": list(ARCHITECTURE_PRESETS.keys()),
            "is_ordered": False,
        },
        {
            "name": "dropout_rate",
            "type": "range",
            "bounds": [0.0, 0.5],
        },
        {
            "name": "activation",
            "type": "choice",
            "values": ["relu", "leaky_relu", "elu", "selu", "gelu"],
        },
        {
            "name": "normalization",
            "type": "choice",
            "values": ["none", "batch_norm", "layer_norm"],
        },
        {
            "name": "weight_decay",
            "type": "range",
            "bounds": [1e-6, 1e-2],
            "log_scale": True,
        },
        {
            "name": "lr",
            "type": "range",
            "bounds": [1e-5, 1e-2],
            "log_scale": True,
        },
        {
            "name": "optimizer_type",
            "type": "choice",
            "values": ["adam", "adamw"],
        },
        {
            "name": "batch_size",
            "type": "choice",
            "values": [32, 64, 128, 256],
        },
        {
            "name": "early_stopping_patience",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)

def evaluate_for_ax(parameterization):
    """Wrapper to convert Ax parameters to evaluate_parameters_nn format."""
    # Get hidden_layers from architecture preset
    architecture = parameterization["architecture"]
    hidden_layers = ARCHITECTURE_PRESETS[architecture]
    
    # Handle normalization choice
    normalization = parameterization["normalization"]
    use_batch_norm = normalization == "batch_norm"
    use_layer_norm = normalization == "layer_norm"
    
    # Handle dropout type based on activation
    activation = parameterization["activation"]
    dropout_type = "alpha" if activation == "selu" else "standard"
    
    # Build parameters dict
    parameters = {
        "hidden_layers": hidden_layers,
        "dropout_rate": parameterization["dropout_rate"],
        "dropout_type": dropout_type,
        "activation": activation,
        "use_batch_norm": use_batch_norm,
        "use_layer_norm": use_layer_norm,
        "weight_decay": parameterization["weight_decay"],
        "lr": parameterization["lr"],
        "optimizer_type": parameterization["optimizer_type"],
        "batch_size": parameterization["batch_size"],
        "early_stopping_patience": parameterization["early_stopping_patience"],
    }
    
    # Run evaluation
    results = evaluate_parameters_nn(parameters, verbose=False)
    
    # Calculate SEM (Standard Error of the Mean) from fold results
    # SEM = std / sqrt(n_folds)
    rmse_nonzero_values = [r["rmse_nonzero"] for r in results["fold_results"]]
    mean_rmse_nonzero = np.mean(rmse_nonzero_values)
    std_rmse_nonzero = np.std(rmse_nonzero_values, ddof=1)  # ddof=1 for sample std
    sem_rmse_nonzero = std_rmse_nonzero / np.sqrt(len(rmse_nonzero_values))
    
    # Return the objective with proper SEM for Bayesian optimization
    return {"avg_rmse_nonzero": (mean_rmse_nonzero, sem_rmse_nonzero)}

[INFO 02-17 08:41:18] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
[INFO 02-17 08:41:18] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter architecture. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
c:\Users\Chris\pytorch_gpu\Lib\site-packages\ax\service\utils\instantiation.py:258: AxParameterWarning: `sort_values` is not specified for `ChoiceParameter` "architecture". Defaulting to `False` for parameters of `ParameterType` STRING. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  return ChoiceParameter(
[INFO 02-17 08:41:18] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter dropout_rate. If that is not the

In [ ]:
# Run the Bayesian optimization loop
n_trials = 100  # Adjust based on your time budget

for i in range(n_trials):
    print(f"\n{'='*60}")
    print(f"Trial {i+1}/{n_trials}")
    print('='*60)
    
    parameters, trial_index = ax_client.get_next_trial()
    print(f"Parameters: {parameters}")
    
    try:
        result = evaluate_for_ax(parameters)
        ax_client.complete_trial(trial_index=trial_index, raw_data=result)
        print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
    except Exception as e:
        print(f"Trial failed: {e}")
        ax_client.log_trial_failure(trial_index=trial_index)

# Get best parameters
best_parameters, values = ax_client.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

In [ ]:

# Get best parameters
best_parameters, values = ax_client.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

#Best parameters: {'dropout_rate': 0.2551544178277254, 'weight_decay': 9.945539915060379e-06, 'lr': 0.002343417645870794, 'optimizer_type': 'adamw', 'batch_size': 256, 'early_stopping_patience': 32, 'architecture': 'constant_2048', 'activation': 'elu', 'normalization': 'layer_norm'}
#Best Non-zero RMSE: 5.2790


In [ ]:
ax_client_2 = AxClient()
ax_client_2.create_experiment(
    name="XGB opt CBFV f(T) Calphed",
    parameters=[
        {
            "name": "n_estimators",
            "type": "range",
            "bounds": [100, 10000],
            "value_type": "int",
        },
        {
            "name": "max_depth",
            "type": "range",
            "bounds": [3, 100],
            "value_type": "int",
        },
        {
            "name": "learning_rate",
            "type": "range",
            "bounds": [0.005, 0.3],
            "value_type": "float",
            "log_scale": True,
        },
        {
            "name": "subsample",
            "type": "range",
            "bounds": [0.5, 1.0],
            "value_type": "float",

        },
        {
            "name": "colsample_bytree",
            "type": "range",
            "bounds": [0.3, 1.0],
            "value_type": "float",
        },
        {
            "name": "min_child_weight",
            "type": "range",
            "bounds": [1, 20],
            "value_type": "int",
        },
        {
            "name": "reg_alpha",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "value_type": "float",
            "log_scale": True,
        },
        {
            "name": "reg_lambda",
            "type": "range",
            "bounds": [1e-6, 10.0],
            "value_type": "float",
            "log_scale": True,
        },
        {
            "name": "gamma",
            "type": "range",
            "bounds": [1e-6, 5.0],
            "log_scale": True,
            "value_type": "float",
        },
        {
            "name": "early_stopping_rounds",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)

def evaluate_for_ax_xgb(parameterization):
    """Wrapper to convert Ax parameters to evaluate_parameters_xgb format."""
    parameters = {
        "n_estimators": parameterization["n_estimators"],
        "max_depth": parameterization["max_depth"],
        "learning_rate": parameterization["learning_rate"],
        "subsample": parameterization["subsample"],
        "colsample_bytree": parameterization["colsample_bytree"],
        "min_child_weight": parameterization["min_child_weight"],
        "reg_alpha": parameterization["reg_alpha"],
        "reg_lambda": parameterization["reg_lambda"],
        "gamma": parameterization["gamma"],
        "early_stopping_rounds": parameterization["early_stopping_rounds"],
    }

    # Run evaluation
    results = evaluate_parameters_xgb(parameters, verbose=False)

    # Calculate SEM (Standard Error of the Mean) from fold results
    rmse_nonzero_values = [r["rmse_nonzero"] for r in results["fold_results"]]
    mean_rmse_nonzero = np.mean(rmse_nonzero_values)
    std_rmse_nonzero = np.std(rmse_nonzero_values, ddof=1)
    sem_rmse_nonzero = std_rmse_nonzero / np.sqrt(len(rmse_nonzero_values))

    return {"avg_rmse_nonzero": (mean_rmse_nonzero, sem_rmse_nonzero)}

[INFO 04-13 09:50:04] ax.generation_strategy.dispatch_utils: Using Generators.BOTORCH_MODULAR since there is at least one ordered parameter and there are no unordered categorical parameters.
[INFO 04-13 09:50:04] ax.generation_strategy.dispatch_utils: Using Bayesian Optimization generation strategy: GenerationStrategy(name='Sobol+BoTorch', steps=[Sobol for 20 trials, BoTorch for subsequent trials]). Iterations after 20 will take longer to generate due to model-fitting.


In [ ]:
# Run the Bayesian optimization loop
n_trials = 100  # Adjust based on your time budget

for i in range(n_trials):
    print(f"\n{'='*60}")
    print(f"Trial {i+1}/{n_trials}")
    print('='*60)
    
    parameters, trial_index = ax_client_2.get_next_trial()
    print(f"Parameters: {parameters}")
    
    try:
        result = evaluate_for_ax_xgb(parameters)
        ax_client_2.complete_trial(trial_index=trial_index, raw_data=result)
        print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
    except Exception as e:
        print(f"Trial failed: {e}")
        ax_client_2.log_trial_failure(trial_index=trial_index)
    
    # Save after each trial for safety
    ax_client_2.save_to_json_file(r"Ax_checkpoints\ax_client_XGB_CALPHAD_V2.json")

# Get best parameters
best_parameters, values = ax_client_2.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

[INFO 03-31 11:02:49] ax.service.ax_client: Generated new trial 0 with parameters {'n_estimators': 5386, 'max_depth': 49, 'learning_rate': 0.17755, 'subsample': 0.88703, 'colsample_bytree': 0.832737, 'min_child_weight': 18, 'reg_alpha': 3.47402, 'reg_lambda': 2.3e-05, 'gamma': 0.003317, 'early_stopping_rounds': 47} using model Sobol.



Trial 1/100
Parameters: {'n_estimators': 5386, 'max_depth': 49, 'learning_rate': 0.17754980241486257, 'subsample': 0.8870299756526947, 'colsample_bytree': 0.8327370762825013, 'min_child_weight': 18, 'reg_alpha': 3.474019734047349, 'reg_lambda': 2.255409678626809e-05, 'gamma': 0.0033168085311114583, 'early_stopping_rounds': 47}

Fold 0
Train: 29906, Val: 7477, Test: 7599
Overall (all)      - RMSE: 1.2994, MAE: 0.3075
Overall (non-zero) - RMSE: 4.2944, MAE: 2.7694  [509,534 values]
Driving Force (DF) - RMSE: 4.3419, MAE: 2.8241  [498,341 non-zero values]
Phase Fraction (NF) - RMSE: 0.4128, MAE: 0.3335  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Overall (all)      - RMSE: 1.4500, MAE: 0.2251
Overall (non-zero) - RMSE: 5.8149, MAE: 3.1232  [1,235,656 values]
Driving Force (DF) - RMSE: 5.8814, MAE: 3.1869  [1,207,679 non-zero values]
Phase Fraction (NF) - RMSE: 0.4722, MAE: 0.3713  [27,977 non-zero values]

Fold 2
Train: 30885, Val: 7722, Test: 6375
Overall (all)

[INFO 03-31 12:34:03] ax.service.ax_client: Completed trial 0 with data: {'avg_rmse_nonzero': 4.555084}.
[INFO 03-31 12:34:03] ax.service.ax_client: Generated new trial 1 with parameters {'n_estimators': 2676, 'max_depth': 55, 'learning_rate': 0.010682, 'subsample': 0.653603, 'colsample_bytree': 0.596856, 'min_child_weight': 4, 'reg_alpha': 0.000398, 'reg_lambda': 7.140359, 'gamma': 5e-06, 'early_stopping_rounds': 17} using model Sobol.


Result: Non-zero RMSE = 4.5551

Trial 2/100
Parameters: {'n_estimators': 2676, 'max_depth': 55, 'learning_rate': 0.010682163418345929, 'subsample': 0.6536027397960424, 'colsample_bytree': 0.5968559330329299, 'min_child_weight': 4, 'reg_alpha': 0.0003979601286139524, 'reg_lambda': 7.140358784129373, 'gamma': 4.510620865069611e-06, 'early_stopping_rounds': 17}

Fold 0
Train: 29906, Val: 7477, Test: 7599


In [ ]:
# Load the XGBoost Ax client from last save and continue optimization to 100 trials if interrupted
ax_client_2 = AxClient.load_from_json_file(r"Ax_checkpoints\ax_client_XGB_CALPHAD_V2.json")

completed_trials = len(ax_client_2.experiment.trials)
target_trials = 100
remaining_trials = target_trials - completed_trials

print(f"Loaded {completed_trials} completed trials from ax_client_xgb.json")
print(f"Remaining trials to reach {target_trials}: {remaining_trials}")

if remaining_trials > 0:
    for i in range(remaining_trials):
        current_trial = completed_trials + i + 1
        print(f"\n{'='*60}")
        print(f"Trial {current_trial}/{target_trials}")
        print('='*60)

        parameters, trial_index = ax_client_2.get_next_trial()
        print(f"Parameters: {parameters}")

        try:
            result = evaluate_for_ax_xgb(parameters)
            ax_client_2.complete_trial(trial_index=trial_index, raw_data=result)
            print(f"Result: Non-zero RMSE = {result['avg_rmse_nonzero'][0]:.4f}")
        except Exception as e:
            print(f"Trial failed: {e}")
            ax_client_2.log_trial_failure(trial_index=trial_index)

        # Save after each trial for safety
        ax_client_2.save_to_json_file(r"Ax_checkpoints\ax_client_XGB_CALPHAD_V2.json")

    # Get best parameters after all trials
    best_parameters, values = ax_client_2.get_best_parameters()
    print(f"\n{'='*60}")
    print("OPTIMIZATION COMPLETE")
    print('='*60)
    print(f"Best parameters: {best_parameters}")
    print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")
else:
    print("Already at or past 100 trials. No additional trials needed.")
    best_parameters, values = ax_client_2.get_best_parameters()
    print(f"Best parameters: {best_parameters}")
    print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

Loaded 35 completed trials from ax_client_xgb.json
Remaining trials to reach 100: 65

Trial 36/100


[INFO 04-13 09:50:19] ax.service.ax_client: Generated new trial 35 with parameters {'n_estimators': 9456, 'max_depth': 5, 'learning_rate': 0.005, 'subsample': 1.0, 'colsample_bytree': 0.310728, 'min_child_weight': 20, 'reg_alpha': 0.04491, 'reg_lambda': 2.790471, 'gamma': 0.146352, 'early_stopping_rounds': 15} using model BoTorch.


Parameters: {'n_estimators': 9456, 'max_depth': 5, 'learning_rate': 0.005, 'subsample': 1.0, 'colsample_bytree': 0.3107278705294097, 'min_child_weight': 20, 'reg_alpha': 0.044910442800718056, 'reg_lambda': 2.7904714339103442, 'gamma': 0.14635229412955159, 'early_stopping_rounds': 15}

Fold 0
Train: 29906, Val: 7477, Test: 7599
Overall (all)      - RMSE: 1.1387, MAE: 0.2974
Overall (non-zero) - RMSE: 4.2275, MAE: 2.6010  [509,534 values]
Driving Force (DF) - RMSE: 4.2744, MAE: 2.6528  [498,341 non-zero values]
Phase Fraction (NF) - RMSE: 0.3671, MAE: 0.2964  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Overall (all)      - RMSE: 1.3324, MAE: 0.2142
Overall (non-zero) - RMSE: 5.4892, MAE: 2.7502  [1,235,656 values]
Driving Force (DF) - RMSE: 5.5521, MAE: 2.8069  [1,207,679 non-zero values]
Phase Fraction (NF) - RMSE: 0.3827, MAE: 0.3028  [27,977 non-zero values]

Fold 2
Train: 30885, Val: 7722, Test: 6375
Overall (all)      - RMSE: 0.7946, MAE: 0.1531
Overall (no

[INFO 04-13 17:31:33] ax.service.ax_client: Completed trial 35 with data: {'avg_rmse_nonzero': 4.153942}.


Result: Non-zero RMSE = 4.1539

Trial 37/100


...

Parameters: {'n_estimators': 9192, 'max_depth': 100, 'learning_rate': 0.21376933665393177, 'subsample': 0.5, 'colsample_bytree': 0.3, 'min_child_weight': 20, 'reg_alpha': 10.0, 'reg_lambda': 1e-06, 'gamma': 5.0, 'early_stopping_rounds': 10}

Fold 0
Train: 29906, Val: 7477, Test: 7599
Overall (all)      - RMSE: 1.1808, MAE: 0.2868
Overall (non-zero) - RMSE: 4.5000, MAE: 2.7595  [509,534 values]
Driving Force (DF) - RMSE: 4.5498, MAE: 2.8139  [498,341 non-zero values]
Phase Fraction (NF) - RMSE: 0.4280, MAE: 0.3348  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Overall (all)      - RMSE: 1.3950, MAE: 0.2269
Overall (non-zero) - RMSE: 5.6148, MAE: 2.8728  [1,235,656 values]
Driving Force (DF) - RMSE: 5.6791, MAE: 2.9321  [1,207,679 non-zero values]
Phase Fraction (NF) - RMSE: 0.4036, MAE: 0.3152  [27,977 non-zero values]

Fold 2
Train: 30885, Val: 7722, Test: 6375
Overall (all)      - RMSE: 0.8938, MAE: 0.1737
Overall (non-zero) - RMSE: 3.5760, MAE: 1.8407  [410,10

[INFO 04-16 12:53:45] ax.service.ax_client: Completed trial 49 with data: {'avg_rmse_nonzero': 4.386176}.


Result: Non-zero RMSE = 4.3862

Trial 51/100


[INFO 04-16 12:54:08] ax.service.ax_client: Generated new trial 50 with parameters {'n_estimators': 5969, 'max_depth': 3, 'learning_rate': 0.005, 'subsample': 0.519308, 'colsample_bytree': 0.325311, 'min_child_weight': 1, 'reg_alpha': 10.0, 'reg_lambda': 1e-06, 'gamma': 0.000967, 'early_stopping_rounds': 33} using model BoTorch.


Parameters: {'n_estimators': 5969, 'max_depth': 3, 'learning_rate': 0.005, 'subsample': 0.5193079895836811, 'colsample_bytree': 0.32531093938916716, 'min_child_weight': 1, 'reg_alpha': 10.0, 'reg_lambda': 1e-06, 'gamma': 0.0009667395542122027, 'early_stopping_rounds': 33}

Fold 0
Train: 29906, Val: 7477, Test: 7599
Overall (all)      - RMSE: 1.1918, MAE: 0.3009
Overall (non-zero) - RMSE: 4.3961, MAE: 2.7713  [509,534 values]
Driving Force (DF) - RMSE: 4.4448, MAE: 2.8268  [498,341 non-zero values]
Phase Fraction (NF) - RMSE: 0.3826, MAE: 0.3028  [11,193 non-zero values]

Fold 1
Train: 23174, Val: 5794, Test: 16014
Overall (all)      - RMSE: 1.3262, MAE: 0.2146
Overall (non-zero) - RMSE: 5.4806, MAE: 2.7266  [1,235,656 values]
Driving Force (DF) - RMSE: 5.5434, MAE: 2.7830  [1,207,679 non-zero values]
Phase Fraction (NF) - RMSE: 0.3812, MAE: 0.2940  [27,977 non-zero values]

Fold 2
Train: 30885, Val: 7722, Test: 6375
Overall (all)      - RMSE: 0.7992, MAE: 0.1549
Overall (non-zero) - RM

In [ ]:
#load the ax client from last save and get best parameters
ax_client_2 = AxClient.load_from_json_file(r"Ax_checkpoints\ax_client_XGB_CALPHAD_V2.json")


# Get best parameters
best_parameters, values = ax_client_2.get_best_parameters()
print(f"\n{'='*60}")
print("OPTIMIZATION COMPLETE")
print('='*60)
print(f"Best parameters: {best_parameters}")
print(f"Best Non-zero RMSE: {values[0]['avg_rmse_nonzero']:.4f}")

[ERROR 03-31 10:51:48] ax.storage.json_store.decoders: Transform LogIntToFloat has been deprecated and removed from Ax. We are unable to load this transform and will return the base `Transform` class instead. The models on the loaded generation strategy may not work correctly!
NoneType: None
[ERROR 03-31 10:51:48] ax.storage.json_store.decoders: Transform LogIntToFloat has been deprecated and removed from Ax. We are unable to load this transform and will return the base `Transform` class instead. The models on the loaded generation strategy may not work correctly!
NoneType: None
[ERROR 03-31 10:51:48] ax.storage.json_store.decoders: Transform LogIntToFloat has been deprecated and removed from Ax. We are unable to load this transform and will return the base `Transform` class instead. The models on the loaded generation strategy may not work correctly!
NoneType: None
[ERROR 03-31 10:51:48] ax.storage.json_store.decoders: Transform LogIntToFloat has been deprecated and removed from Ax. W


OPTIMIZATION COMPLETE
Best parameters: {'n_estimators': 9856, 'max_depth': 25, 'learning_rate': 0.005797910076787395, 'subsample': 0.5037925727665424, 'colsample_bytree': 0.4406051756814122, 'min_child_weight': 11, 'reg_alpha': 0.0010842902645472655, 'reg_lambda': 0.0007034128402734389, 'gamma': 1.3193826703354716, 'early_stopping_rounds': 49}
Best Non-zero RMSE: 4.3023
